# Track A on Colab GPU — NLI scoring + first submission

Runs the Track A pipeline from the repo on a GPU runtime:
1. mounts Drive (data lives in `Bengali-LLM-Hallucination-Detection/data/` there)
2. clones/pulls the GitHub repo
3. scores train + test rows with the NLI model (minutes on a T4, ~1h on CPU)
4. builds + validates `submissions/nli_ctx_majority_cb.csv`
5. copies the score caches and the submission back to Drive

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Bengali-LLM-Hallucination-Detection'
import os
assert os.path.exists(f'{DRIVE_DIR}/data/test set.csv'), 'competition data not found on Drive'

In [ ]:
%cd /content
!git clone https://github.com/FaiyazAwsaf/Bengali-LLM-Hallucination-Detection.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo

# data/ is gitignored, so bring it in from Drive
!mkdir -p data submissions
!cp "{DRIVE_DIR}/data/dataset samples.json" "{DRIVE_DIR}/data/test set.csv" "{DRIVE_DIR}/data/sample submission.csv" data/

# reuse previously computed score caches from Drive if they exist
!mkdir -p data/cache
!cp "{DRIVE_DIR}/cache/"*.csv data/cache/ 2>/dev/null || echo 'no cache on Drive yet'

!pip install -q -U transformers sentencepiece

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable a GPU runtime')

In [ ]:
# score the 299 labeled rows + run repeated CV (fast; verifies everything works)
!cd src && python track_a_nli.py cv

In [ ]:
# score the 1,361 test context rows (the expensive step — minutes on T4)
!cd src && python track_a_nli.py test | tail -3

In [ ]:
# build + validate the submission CSV
!cd src && python combine_and_predict.py nli_ctx_majority_cb.csv

In [ ]:
# persist results to Drive: score caches (reusable) + the submission file
!mkdir -p "{DRIVE_DIR}/cache" "{DRIVE_DIR}/submissions"
!cp data/cache/nli_*.csv "{DRIVE_DIR}/cache/"
!cp submissions/nli_ctx_majority_cb.csv "{DRIVE_DIR}/submissions/"
!ls -la "{DRIVE_DIR}/cache" "{DRIVE_DIR}/submissions"

Done. Back on the local machine, download the caches + submission from Drive into `data/cache/`
and `submissions/`, then submit with:

```
kaggle competitions submit -c <competition> -f submissions/nli_ctx_majority_cb.csv -m "Track A NLI + majority closed-book"
```